In [1]:
include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")

evaluate (generic function with 1 method)

## 1. Prepare Environment

In [2]:
using RockSample

pomdp = RockSamplePOMDP(7, 8)
pomdp_name = "RS78"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# define convert_o function
function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::RockSamplePOMDP)
    vec = zeros(Float32, 3)
    vec[o] = 1.0f0
    return vec
end

# define process action function
function process_action(action::Int, action_space::UnitRange{Int})
    len = length(action_space)
    idx = action - first(action_space) + 1
    (idx < 1 || idx > len) && error("Action $action not in action space")
    onehot = zeros(Float32, len)
    onehot[idx] = 1.0f0
    return onehot
end

process_action (generic function with 1 method)

## 2. Prepare Parameters

In [3]:
state_dim = GetObsDim(env)
layer_size = 64
rnn_hidden_size = 64
gamma = discount(pomdp)
training_episodes = 10000
batch_size = 1024

RSState{8}([2, 1], Bool[1, 0, 0, 0, 0, 1, 1, 1])
Float32[0.0, 0.0, 1.0]


1024

## 3. Prepare PPO-RNN agent

In [4]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
# using CUDA

agent = PPORNNAgent(action_space, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.cpu) 

PPORNNAgent(Chain(LSTM(16 => 64), Dense(64 => 64, tanh), Dense(64 => 13)), Chain(Dense(16 => 64, tanh), LSTM(64 => 64), Dense(64 => 64, tanh), Dense(64 => 1)), (layers = ((cell = (Wi = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999)))

## 4. Train

In [ ]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

Progress:   1%|█                                        |  ETA: 10:27:5427m

## 5. Evaluation

In [ ]:
include("./Algorithms/PPO-RNN.jl")
evaluate(env, agent; num_episodes=10000, max_steps=100) 

## (Todo) Save or plot the data from Train (rewards, losses, evals)